In [ ]:
#| default_exp icon4

# icon4
> Jupyter Lab embedded in the visualization area (``ST_JUPYTER_LAB_URL``). Notebook REST appends and chat unchanged.

The viz slot is a full-size **iframe** loading the URL from ``ST_JUPYTER_LAB_URL`` (see ``therapy_config.jupyter_lab_link_url()``). Jupyter Server must allow embedding (e.g. CSP ``frame-ancestors`` including your app origin). **`append_chat_turn_to_notebook`** uses the REST API as before.

In [ ]:
#| export
from __future__ import annotations

import json
import os
from urllib.parse import quote

import httpx
import nbformat
from fastcore.xml import to_xml
from fasthtml.common import *

import string_therapy.therapy_config as tc
from string_therapy.lisette_client import lisette_chat

## Constants

`ICON_ID` identifies this slot for routing or logging. `SESSION_NOTEBOOK_KEY` is where the FastHTML session stores the **relative path** of the notebook file Lab should open (e.g. `my_notes.ipynb` or a path under the server root).

In [ ]:
#| export
ICON_ID = 4
SESSION_NOTEBOOK_KEY = "icon4_notebook_path"

## Jupyter path helpers

**`ensure_icon4_sid`** hands chat (`lisette_chat`) a stable id per browser session. **`default_notebook_path`** picks the file when nothing is stored yet (`JUPYTER_NOTEBOOK_PATH` or `my_notes.ipynb`). **`_contents_api_url`** is the REST base for reading/writing notebook JSON.

**`resolve_notebook_path`** prefers a safe, non-traversal `notebook` query string when present, then the session value, then the default. **`session_notebook_path`** is “current path” without a request override.

In [ ]:
#| export
def ensure_icon4_sid(session: dict) -> str:
    """Return a stable per-session id for icon4; create and store it if missing."""
    k = "icon4_sid"
    if k not in session:
        session[k] = os.urandom(8).hex()
    return session[k]


def default_notebook_path() -> str:
    """Notebook file path when none is stored: ``JUPYTER_NOTEBOOK_PATH`` or ``my_notes.ipynb``."""
    return (os.getenv("JUPYTER_NOTEBOOK_PATH") or "my_notes.ipynb").strip()


def _jupyter_base_url() -> str:
    """HTTP origin for Jupyter server."""
    return tc.jupyter_base_url()


def _encode_nb_path(notebook_path: str) -> str:
    """URL-encode each path segment for Jupyter Server ``/api/contents/...`` URLs."""
    parts = [p for p in notebook_path.split("/") if p]
    return "/".join(quote(p, safe="") for p in parts)


def _contents_api_url(notebook_path: str) -> str:
    """Jupyter Server REST URL for GET/PUT notebook JSON (``/api/contents/...``)."""
    return f"{_jupyter_base_url()}/api/contents/{_encode_nb_path(notebook_path)}"


def resolve_notebook_path(notebook: str | None, session: dict) -> str:
    """Pick notebook path: non-empty ``notebook`` param if safe, else session, else ``default_notebook_path()``."""
    if notebook and isinstance(notebook, str):
        p = notebook.strip().replace("\\", "/").strip("/")
        parts = [x for x in p.split("/") if x]
        if p and ".." not in parts:
            return p
    stored = session.get(SESSION_NOTEBOOK_KEY)
    if isinstance(stored, str) and stored.strip():
        return stored.strip().replace("\\", "/").strip("/") or default_notebook_path()
    return default_notebook_path()


def session_notebook_path(session: dict) -> str:
    """Active notebook path from session (and defaults), ignoring a request override."""
    return resolve_notebook_path(None, session)


def lab_iframe_src(*, cache_bust: str | None = None) -> str:
    """Iframe ``src``: exact URL from ``jupyter_lab_link_url()`` (``ST_JUPYTER_LAB_URL`` in ``.env``)."""
    u = tc.jupyter_lab_link_url()
    if not cache_bust:
        return u
    sep = "&" if "?" in u else "?"
    return f"{u}{sep}_app_ts={cache_bust}"

## Jupyter Lab iframe (HTMX out-of-band)

**`lab_iframe_src()`** is exactly **`ST_JUPYTER_LAB_URL`**. **`jupyter_wrap_oob`** replaces **`#jupyter-wrap`** (`hx_swap_oob`).

In [ ]:
#| export
_JUPYTER_WRAP_CLS = (
    "relative w-full flex-1 min-h-0 overflow-hidden rounded-lg border border-base-300 bg-black"
)

_JUPYTER_IFRAME_ALLOW = (
    "accelerometer; autoplay; clipboard-read; clipboard-write; display-capture; "
    "encrypted-media; fullscreen; geolocation; microphone; picture-in-picture; web-share"
)


def jupyter_wrap_oob(session: dict, *, cache_bust: str | None = None) -> Div:
    """OOB fragment: replaces #jupyter-wrap with the Lab iframe."""
    _ = session
    return Div(
        Iframe(
            src=lab_iframe_src(cache_bust=cache_bust),
            cls="absolute inset-0 h-full w-full border-0",
            id="jupyter-main",
            hx_preserve="true",
            allow=_JUPYTER_IFRAME_ALLOW,
        ),
        id="jupyter-wrap",
        cls=_JUPYTER_WRAP_CLS,
        hx_swap_oob="true",
    )

## Visualization and chat

**`viz`** is only the Lab **iframe**. **`chat`** / **`get_oob_html`** unchanged.

In [ ]:
#| export
def viz(sid: str, session: dict):
    """Icon 4: Jupyter Lab iframe (``ST_JUPYTER_LAB_URL``)."""
    path = session_notebook_path(session)
    session.setdefault(SESSION_NOTEBOOK_KEY, path)
    return Div(
        Div(
            Iframe(
                src=lab_iframe_src(),
                cls="absolute inset-0 h-full w-full border-0",
                id="jupyter-main",
                hx_preserve="true",
                allow=_JUPYTER_IFRAME_ALLOW,
            ),
            id="jupyter-wrap",
            cls=_JUPYTER_WRAP_CLS,
        ),
        cls="flex flex-col flex-1 min-h-0 w-full overflow-hidden",
    )


def chat(sid: str, msg: str, max_tokens: int = 2000):
    """Run Lisette chat for this visualization; notebook writes are handled in ``app.py``."""
    return lisette_chat(sid, msg, max_tokens=max_tokens)


def get_oob_html(session: dict, sid: str):
    """Return an empty trigger plus serialized ``viz`` XML for out-of-band shell refresh."""
    return "", to_xml(viz(sid, session=session), indent=False)

## Appending chat turns to the notebook (Jupyter Server API)

**`append_chat_turn_to_notebook`** is **async**: it GETs the notebook document from **`_contents_api_url`**, parses it with **nbformat**, appends cells, then PUTs the updated JSON. Callers (e.g. **`app.py`**) use this after **`chat`** returns; refresh the notebook in Jupyter Lab to see new cells.

In [ ]:
#| export
async def append_chat_turn_to_notebook(
    notebook_path: str,
    user_message: str,
    reply_cell_type: str,
    assistant_text: str | None,
) -> tuple[bool, str]:
    """Append user prompt as markdown, then assistant reply as markdown or code (one GET + one PUT)."""
    api_url = _contents_api_url(notebook_path)
    headers: dict[str, str] = {}
    tok = (os.getenv("JUPYTER_TOKEN") or "").strip()
    if tok:
        headers["Authorization"] = f"token {tok}"

    um = (user_message or "").strip()
    if not um:
        return False, "Nothing to append."

    rt = (reply_cell_type or "markdown").strip().lower()
    if rt not in ("markdown", "code"):
        rt = "markdown"

    at = (assistant_text or "").strip() or None

    async with httpx.AsyncClient(timeout=60.0) as client:
        try:
            res = await client.get(api_url, headers=headers)
            if res.status_code != 200:
                return False, f"Could not read notebook ({res.status_code})."
            nb_data = res.json()
            raw = nb_data.get("content")
            if not isinstance(raw, dict):
                return False, "Invalid notebook JSON."

            nb = nbformat.reads(json.dumps(raw), as_version=4)
            nb.cells.append(nbformat.v4.new_markdown_cell(um))
            if at:
                if rt == "code":
                    nb.cells.append(nbformat.v4.new_code_cell(at))
                else:
                    nb.cells.append(nbformat.v4.new_markdown_cell(at))

            try:
                nbformat.validate(nb)
            except Exception:
                pass

            new_content = json.loads(nbformat.writes(nb))
            save_res = await client.put(
                api_url,
                headers=headers,
                json={"type": "notebook", "content": new_content},
            )
            if save_res.status_code == 200:
                if at:
                    return True, "Cells added."
                return True, "Prompt added (no AI reply)."
            return False, f"Save failed ({save_res.status_code})."
        except Exception as e:
            return False, f"Jupyter: {e}"

## Routes

**`register_routes`** attaches **`GET /icon4/jupyter-frame`** (iframe fragment + session notebook path).

In [ ]:
#| export
def register_routes(app):
    """Attach icon4 routes: HTMX partial for the Lab iframe + notebook path."""

    @app.get("/icon4/jupyter-frame")
    def icon4_jupyter_frame(session, notebook: str | None = None):
        """Return the Lab iframe; updates session notebook path."""
        path = resolve_notebook_path(notebook, session)
        session[SESSION_NOTEBOOK_KEY] = path
        return Div(
            Iframe(
                src=lab_iframe_src(),
                cls="absolute inset-0 h-full w-full border-0",
                id="jupyter-main",
                hx_preserve="true",
                allow=_JUPYTER_IFRAME_ALLOW,
            ),
            id="jupyter-wrap",
            cls=_JUPYTER_WRAP_CLS,
        )